# Pinky 라이브러리 LLM fine tuning하기

이 노트북은 강의자료 `Pinky 라이브러리 LLM fine tuning하기` 1개에 대응하는 통합 실습 노트북입니다.

구성:
1. `pinky_finetune_data.json` 학습 데이터 준비
2. Unsloth + LoRA 기반 파인튜닝
3. GGUF 저장 및 Ollama 등록

실행 전 CUDA 사용 가능한 GPU 환경과 `pinky_library.md` 파일을 준비하세요.

## 1. 파인튜닝 전 데이터 생성

강의자료의 1장에 해당합니다. `pinky_library.md`를 참고하여 `pinky_finetune_data.json` 학습 데이터를 준비합니다.

In [ ]:
from pathlib import Path
import json

DATA_PATH = Path("pinky_finetune_data.json")
MD_PATH = Path("pinky_library.md")

print("pinky_library.md 존재 여부:", MD_PATH.exists())
print("pinky_finetune_data.json 존재 여부:", DATA_PATH.exists())

## 데이터 생성용 프롬프트

```text
첨부한 `pinky_library.md` 파일은 'Pinky'라는 교육용 로봇을 제어하기 위한 파이썬 라이브러리 명세서입니다.
이 문서를 완벽하게 이해한 뒤, 로봇 제어용 LLM(거대언어모델)을 파인튜닝하기 위한 학습 데이터를 생성해주세요.

### [요구사항]
1. 포맷: 반드시 아래와 같은 JSON 리스트 형식으로 작성해주세요.
   [
     {"input": "사용자 질문", "output": "실행 가능한 파이썬 코드"}
   ]
2. 내용 구성: 총 20개의 다양한 예제를 만들어주세요.
   - 단순 제어 (예: "앞으로 1초 이동해")
   - 센서 활용 (예: "장애물 있으면 멈춰")
   - 복합 로직 (예: "카메라로 마커 찾아서 ID 출력해")
   - 로봇 설명 (예: "초음파 센서는 어떻게 써?")
3. 코드 작성 규칙:
   - 반드시 첨부된 문서에 정의된 클래스와 함수만 사용하세요. 없는 함수 창조 금지.
   - 하드웨어 안전을 위해 try-finally 블록을 사용하여 마지막에 꼭 close()를 호출하도록 작성하세요.
   - 주석과 설명은 한국어로 작성하세요.
4. 출력: 서론이나 사족 없이 오직 JSON 데이터만 출력해주세요.
```

데이터가 부족하면 강의자료의 예시처럼 영역별로 추가 요청하면 됩니다.

- 모터 제어 관련 20개 추가
- 센서 조건 제어 20개 추가
- 센서 + 모터 복합 로직 20개 추가
- 카메라 / ArUco 20개 추가
- 설명형 질문 20개 추가

In [ ]:
# JSON 파일 구조 검증용 셀
# pinky_finetune_data.json을 만든 뒤 실행하세요.

if not DATA_PATH.exists():
    print("아직 pinky_finetune_data.json 파일이 없습니다.")
else:
    with DATA_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, list), "최상위 구조는 list여야 합니다."
    for i, item in enumerate(data):
        assert isinstance(item, dict), f"{i}번째 항목은 dict여야 합니다."
        assert "input" in item and "output" in item, f"{i}번째 항목에 input/output 키가 필요합니다."
        assert isinstance(item["input"], str) and item["input"].strip(), f"{i}번째 input이 비어 있습니다."
        assert isinstance(item["output"], str) and item["output"].strip(), f"{i}번째 output이 비어 있습니다."

    print(f"검증 완료: {len(data)}개 학습 데이터")
    print("첫 번째 예시:")
    print(json.dumps(data[0], ensure_ascii=False, indent=2))

## 2. 파인튜닝

강의자료의 `train_pinky.py`에 해당합니다. Unsloth 모델을 로드하고 LoRA 어댑터를 붙인 뒤, Alpaca 포맷 데이터로 지도 미세 조정합니다.

In [ ]:
# 파인튜닝에 필요한 라이브러리 설치
!pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
from pathlib import Path

data_path = Path("pinky_finetune_data.json")
if not data_path.exists():
    raise FileNotFoundError("pinky_finetune_data.json 파일이 필요합니다. 먼저 위의 데이터 생성 섹션에서 데이터를 준비하세요.")

print("데이터 파일 확인:", data_path.resolve())

라이브러리 import와 기본 설정입니다.

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
import json
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configuration
max_seq_length = 2048
dtype = None       # Auto detection
load_in_4bit = True
input_json = "pinky_finetune_data.json"
model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit"  # Using the one from the notebook

모델을 로드합니다.

In [ ]:
# 2. Load Model
print("Loading Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

LoRA 어댑터를 추가합니다.

In [ ]:
# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Adjust rank if needed (16, 32, 64)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

JSON 데이터를 Alpaca 포맷으로 변환합니다.

In [ ]:
# 4. Prepare Dataset
print("Preparing Dataset...")
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

# 시스템 프롬프트 정의
system_prompt = """당신은 교육용 로봇 'Pinky'를 제어하는 Python 전문가입니다.
사용자의 요청에 대해 `pinkylib`과 `pinky_lcd` 라이브러리를 사용하여 정확한 Python 코드를 작성하세요.
모르는 내용이거나 라이브러리에 없는 기능이라면 억지로 코드를 만들지 말고 솔직하게 "모르겠습니다"라고 대답하세요.
답변은 한국어로 친절하게 설명하세요."""

def format_prompt(example: dict) -> str:
    input_text = example["input"]
    output_text = example["output"]

    # 시스템 프롬프트
    system_message = system_prompt

    # 표준 Alpaca 포맷 적용
    text = f"""### Instruction:
{system_message}

{input_text}

### Response:
{output_text}"""

    return text + tokenizer.eos_token

# Convert to HuggingFace Dataset
formatted_data = [{"text": format_prompt(item)} for item in data]
dataset = Dataset.from_list(formatted_data)

print("데이터셋 크기:", len(dataset))
print(dataset[0]["text"][:1000])

학습을 실행합니다. 주피터노트북에서 W&B 선택 입력으로 멈추는 것을 줄이기 위해 `report_to="none"`을 추가했습니다.
강의자료처럼 W&B 선택 화면을 보고 싶다면 `report_to="none"` 줄을 삭제하세요.

In [ ]:
# 5. Train
print("Starting Training...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=10,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="pinky_outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
trainer_stats

학습한 모델을 GGUF로 저장합니다. Ollama에서 사용할 수 있도록 `pinky_model` 폴더에 `q4_k_m` 양자화 파일을 만듭니다.

In [ ]:
# 6. Save Model
print("Saving Model...")
# model.save_pretrained("pinky_lora_model")
# tokenizer.save_pretrained("pinky_lora_model")

# Merge to GGUF
model.save_pretrained_gguf("pinky_model", tokenizer, quantization_method="q4_k_m")
print("Done!")

생성된 GGUF 파일을 확인합니다.

In [ ]:
from pathlib import Path

ggufs = sorted(Path("pinky_model").glob("*.gguf"))
for path in ggufs:
    print(path)

## 3. GGUF 파일 Ollama 등록

파인튜닝 결과로 만들어진 GGUF 파일을 `pinky_llm.gguf`로 준비하고, `Modelfile`을 작성한 뒤 Ollama에 등록합니다.

In [ ]:
from pathlib import Path
import shutil

# 위 학습 섹션에서 생성된 GGUF 파일을 pinky_llm.gguf로 복사/변경합니다.
# 이미 pinky_llm.gguf가 있다면 그대로 둡니다.
target = Path("pinky_llm.gguf")

if target.exists():
    print("이미 pinky_llm.gguf가 있습니다:", target.resolve())
else:
    candidates = sorted(Path("pinky_model").glob("*.gguf"))
    if not candidates:
        raise FileNotFoundError("pinky_model 폴더에서 .gguf 파일을 찾지 못했습니다. 먼저 위 학습 섹션을 실행하세요.")
    source = candidates[0]
    shutil.copy2(source, target)
    print(f"{source} -> {target} 복사 완료")

강의자료의 Modelfile 내용을 생성합니다.

In [ ]:
%%writefile Modelfile
# 1. 모델 경로 설정
FROM ./pinky_llm.gguf

# 2. 템플릿 설정 (Alpaca Style)
TEMPLATE """### Instruction:
{{ .System }}
{{ .Prompt }}
### Response:
"""

# 3. 시스템 프롬프트
SYSTEM """당신은 교육용 로봇 'Pinky'를 제어하는 Python 전문가입니다.
사용자의 요청에 대해 `pinkylib`과 `pinky_lcd` 라이브러리를 사용하여 정확한 Python 코드를 작성하세요.
하드웨어 제어 시에는 반드시 `try-finally` 구문을 사용하여 `close()`로 자원을 해제해야 합니다.
모르는 내용이거나 라이브러리에 없는 기능이라면 억지로 코드를 만들지 말고 솔직하게 "모르겠습니다"라고 대답하세요.
답변은 한국어로 친절하게 설명하세요."""

# 4. 파라미터 튜닝
# 코드 생성 모델이므로 창의성(temperature)을 낮춰 환각을 줄입니다.
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1

In [ ]:
# Ollama에 모델 등록
!ollama create pinky_llm.gguf -f ./Modelfile

In [ ]:
# 등록 확인
!ollama list

In [ ]:
# 실행 테스트
# !ollama run pinky_llm.gguf "Pinky 로봇을 앞으로 1초 이동시키는 코드를 작성해줘"

참고: 강의자료에서는 문제가 생기면 `pinky_outputs`, `pinky_lora_model` 폴더를 삭제하고 다시 학습하는 방법을 안내합니다.

In [ ]:
# 필요할 때만 실행하세요.
# !rm -rf pinky_outputs pinky_lora_model